# Experiment 1.2b-i — Finite-Time Sensitivity to Gauge-Preserving vs. Non-Gauge Perturbations

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/Exp_1.2_Spectrum_Perturbation_Invariance/1.2b_Perturbation_Accumulation_Over_Trajectory/1.2b-i_Lyapunov_Exponent_of_Gauge_Direction/run_experiment.py`

## Truthful scope of this second-pass strengthening

This notebook now imports and runs the shared script implementation directly. The pair therefore shares one computational core.

The study remains a **toy finite-time perturbation sensitivity experiment** for a 4-layer deep linear network trained with either:
- **SGD with momentum**, or
- a **Muon-style** momentum update with Newton–Schulz gradient orthogonalization.

The key strengthening is that the primary `gauge` condition is now a **true coupled gauge-preserving reparameterization at $t=0$**:
- hidden-basis transforms $G_k = I + arepsilon S_k/\|S_k\|_F$ are sampled on the internal interfaces,
- the layers are perturbed as
  - $W_1' = G_1 W_1$,
  - $W_i' = G_i W_i G_{i-1}^{-1}$ for $1 < i < L$,
  - $W_L' = W_L G_{L-1}^{-1}$,
- so the end-to-end map $W_{\mathrm{eff}} = W_L \cdots W_1$ is preserved at initialization up to numerical roundoff.

The pair also retains two comparison conditions:
- `physical`: independent **skew** right-multiplicative control perturbations,
- `symmetric_legacy`: the old independent **symmetric** right-multiplicative perturbation, now explicitly labeled as **non-gauge legacy**.

> **Important caveat:** this is still a finite-time sensitivity study, not a strict asymptotic maximal Lyapunov exponent computation. Only the primary `gauge` condition is function-preserving at $t=0$.

## Notebook structure

1. Resolve paths and import the script safely from a notebook.
2. Log reproducibility information, perturbation definitions, and active configuration.
3. Run the experiment via the script's `run_experiment(...)` API.
4. Verify that the new primary `gauge` perturbation really preserves the effective map at initialization.
5. Summarize $\lambda_W$, $\lambda_F$, and $\lambda_L$ by condition.
6. Review training sanity checks, verdict components, and figures produced by the shared script core.
7. End with a calibrated conclusion based on the measured outputs, not a prewritten theory claim.

In [ ]:
from pathlib import Path
import importlib.util
import os
import platform
import sys
import time

import numpy as np
from IPython.display import Image, Markdown, display

EXPERIMENT_SCRIPT_RELATIVE = Path(
    r"experiments/Tier_1_Core_Mechanism_Tests/Exp_1.2_Spectrum_Perturbation_Invariance/1.2b_Perturbation_Accumulation_Over_Trajectory/1.2b-i_Lyapunov_Exponent_of_Gauge_Direction/run_experiment.py"
)
NOTEBOOK_SMOKE = os.environ.get("LYAPUNOV_GAUGE_NOTEBOOK_SMOKE", "0") == "1"
SMOKE_OVERRIDES = {"num_steps": 20, "num_perturbations": 4}


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / EXPERIMENT_SCRIPT_RELATIVE).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory. "
        f"Expected to find {EXPERIMENT_SCRIPT_RELATIVE} while walking upward from {start}."
    )


def load_experiment_module(script_path: Path):
    spec = importlib.util.spec_from_file_location("exp_1_2b_i_run_experiment", script_path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Unable to load module from {script_path}")
    spec.loader.exec_module(module)
    return module


def format_value(value):
    if isinstance(value, float):
        if np.isnan(value):
            return "nan"
        if np.isinf(value):
            return "inf" if value > 0 else "-inf"
        if abs(value) >= 1e4 or (0 < abs(value) < 1e-3):
            return f"{value:.3e}"
        return f"{value:.6f}"
    return str(value)


def markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(format_value(v) for v in row) + " |")
    return "\n".join(lines)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SCRIPT_PATH = REPO_ROOT / EXPERIMENT_SCRIPT_RELATIVE
EXPERIMENT_DIR = SCRIPT_PATH.parent
exp12bi = load_experiment_module(SCRIPT_PATH)

print(f"Repo root:        {REPO_ROOT}")
print(f"Counterpart file: {SCRIPT_PATH}")
print(f"Experiment dir:   {EXPERIMENT_DIR}")
print(f"Notebook smoke:   {NOTEBOOK_SMOKE}")

## Reproducibility, runtime, and perturbation catalog

This notebook exposes both the default script configuration and the perturbation catalog used by the shared experiment core.

- In normal mode, it runs the default configuration from the script.
- In smoke mode (`LYAPUNOV_GAUGE_NOTEBOOK_SMOKE=1`), it only overrides `num_steps` and `num_perturbations` to make end-to-end notebook execution faster while preserving the same code path.
- The script also performs an SGD learning-rate stability scan before the main conditions.

In [ ]:
default_config = exp12bi.get_default_config()
perturbation_catalog = exp12bi.get_perturbation_catalog()
config_overrides = dict(SMOKE_OVERRIDES) if NOTEBOOK_SMOKE else {}
effective_config = dict(default_config)
effective_config.update(config_overrides)

config_rows = [[key, effective_config[key]] for key in sorted(effective_config.keys())]
display(Markdown(markdown_table(["Config key", "Active value"], config_rows)))

catalog_rows = []
for key, meta in perturbation_catalog.items():
    catalog_rows.append([
        key,
        meta["is_true_gauge"],
        meta["display_label"],
        meta["construction"],
    ])

display(Markdown("### Perturbation catalog"))
display(Markdown(markdown_table(
    ["Key", "True gauge?", "Display label", "Construction"],
    catalog_rows,
)))

print(f"Python version: {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print(f"NumPy version:  {np.__version__}")
print(f"Working dir:    {Path.cwd().resolve()}")
print(f"Config overrides active: {config_overrides if config_overrides else 'none'}")

## Metric definitions and interpretation guide

The script returns structured raw results for each optimizer/perturbation condition. The most important summaries are:

- **$\lambda_W = 	frac{1}{N}\log(d_W(N)/d_W(0))$** where $d_W$ is the Frobenius distance across the stacked layer parameters.
- **$\lambda_F = 	frac{1}{N}\log(d_F(N)/d_F(0))$** where $d_F$ is the Frobenius distance between the end-to-end effective matrices.
- **$\lambda_L$** = mean of the per-layer finite-time growth rates over the same perturbation trial family.

Interpretation remains modest:
- these are **finite-time** growth rates, not asymptotic Lyapunov exponents in a strict dynamical-systems sense;
- uncertainty here is across **perturbation directions** only, not across multiple data or initialization seeds;
- for the true gauge-preserving condition, $d_F(0)$ is expected to be essentially zero, so $\lambda_F$ can become undefined and should be read with care rather than forced into an overclaim.

In [ ]:
run_started = time.time()
results = exp12bi.run_experiment(
    config_overrides=config_overrides,
    output_dir=EXPERIMENT_DIR,
    generate_plots=True,
    verbose=True,
)
run_wall_seconds = time.time() - run_started

print(f"Notebook-observed wall-clock time: {run_wall_seconds:.2f} seconds")
print(f"Script-reported runtime:           {results['runtime_seconds']:.2f} seconds")
print(f"Learning rates used:               {results['learning_rates']}")
print(f"Seeds: global={results['config']['seed_global']}, init={results['config']['seed_init']}, perturb_base={results['config']['seed_perturb_base']}")

## Initial effective-map mismatch diagnostics

The most important scientific strengthening in this pass is that the primary `gauge` condition is now tested **as an actual function-preserving perturbation at initialization**.

The table below uses the script's returned diagnostics to show, for each condition:
- the mean and maximum initial effective-map mismatch,
- the corresponding relative output mismatch and loss gap,
- and whether the condition satisfies the script's preservation tolerance.

For a true gauge-preserving condition, the effective-map mismatch should be at numerical-roundoff scale rather than merely "small.

In [ ]:
diagnostic_rows = []
for condition_key in results["condition_order"]:
    condition = results["condition_results"][condition_key]
    diagnostics = condition["initial_diagnostics"]
    meta = condition["perturbation_metadata"]
    diagnostic_rows.append([
        condition_key,
        meta["is_true_gauge"],
        diagnostics["initial_effective_mismatch_abs"]["mean"],
        diagnostics["max_initial_effective_mismatch_abs"],
        diagnostics["initial_effective_mismatch_rel"]["mean"],
        diagnostics["initial_output_mismatch_rel"]["mean"],
        diagnostics["initial_loss_gap_abs"]["mean"],
        diagnostics["preserves_effective_map_within_tol"],
    ])

display(Markdown(markdown_table(
    [
        "Condition",
        "True gauge?",
        "mean init d_F",
        "max init d_F",
        "mean rel eff mismatch",
        "mean rel output mismatch",
        "mean init loss gap",
        "preserves map within tol?",
    ],
    diagnostic_rows,
)))

verdict = results["verdict"]
gauge_check = verdict["gauge_construction_check"]
display(Markdown(f"""
### Gauge-construction verification

- pass: **{gauge_check['pass']}**
- maximum absolute effective-map mismatch observed across the primary gauge trials: **{gauge_check['max_abs_observed']:.3e}**
- absolute tolerance: **{gauge_check['tolerance_abs']:.1e}**
- maximum relative effective-map mismatch observed across the primary gauge trials: **{gauge_check['max_rel_observed']:.3e}**
- relative tolerance: **{gauge_check['tolerance_rel']:.1e}**

This is the main evidence that the new `gauge` condition is now materially different from the old independent symmetric perturbation.
"""))

## Condition summary: $\lambda_W$, $\lambda_F$, and $\lambda_L$

The next table is assembled directly from the script's returned result dictionary. It includes normal-approximation 95% confidence interval widths (`ci95`) across perturbation directions for each metric summary.

Because the true gauge condition preserves the end-to-end map at initialization, its $\lambda_F$ summary can contain invalid samples when $d_F(0)$ is effectively zero. That behavior is expected and should be read as an honesty feature, not a bug.

In [ ]:
summary_rows = []
for condition_key in results["condition_order"]:
    condition = results["condition_results"][condition_key]
    summary = condition["summary"]
    diagnostics = condition["initial_diagnostics"]
    perturbation_label = condition["perturbation_metadata"]["display_label"]
    summary_rows.append([
        condition_key,
        perturbation_label,
        f"{summary['lambda_W']['mean']:.6f} ± {summary['lambda_W']['ci95']:.6f}",
        f"{summary['lambda_F']['mean']:.6f} ± {summary['lambda_F']['ci95']:.6f}",
        f"{summary['lambda_L']['mean']:.6f} ± {summary['lambda_L']['ci95']:.6f}",
        summary['lambda_F']['n_valid'],
        diagnostics['initial_effective_mismatch_abs']['mean'],
        diagnostics['preserves_effective_map_within_tol'],
    ])

display(Markdown(markdown_table(
    [
        "Condition",
        "Perturbation label",
        "lambda_W (mean ± ci95)",
        "lambda_F (mean ± ci95)",
        "lambda_L (mean ± ci95)",
        "valid lambda_F samples",
        "mean init d_F",
        "map-preserving?",
    ],
    summary_rows,
)))

In [ ]:
primary = results["verdict"]["primary_shortform"]
legacy = results["verdict"]["legacy_shortform"]
interpretation = f"""
### Immediate reading of the measured outputs

**Primary gauge-preserving condition:**
- SGD gauge $\lambda_W$ = **{primary['lambda_gauge_SGD']:+.6f}**
- Muon gauge $\lambda_W$ = **{primary['lambda_gauge_Muon']:+.6f}**

**Skew control condition:**
- SGD skew $\lambda_W$ = **{primary['lambda_phys_SGD']:+.6f}**
- Muon skew $\lambda_W$ = **{primary['lambda_phys_Muon']:+.6f}**

**Retained legacy symmetric comparison:**
- SGD legacy symmetric $\lambda_W$ = **{legacy['lambda_legacy_symmetric_SGD']:+.6f}**
- Muon legacy symmetric $\lambda_W$ = **{legacy['lambda_legacy_symmetric_Muon']:+.6f}**

This means the pair can now separately answer two different questions:
1. what happens under a **true gauge-preserving initialization perturbation**, and
2. how that differs from the old **non-gauge symmetric perturbation** that previously carried the strongest conceptual flaw.
"""

display(Markdown(interpretation))

## Training sanity checks and verdict components

Before interpreting perturbation sensitivity, the pair should still confirm that both optimizers reduce loss under the active setup. The script also returns:
- a gauge-construction verification check,
- primary verdict tests for the true gauge-preserving condition,
- retained legacy verdict tests for comparison,
- and Welch-style comparisons of the SGD-vs-Muon $\lambda_W$ samples.

In [ ]:
training_rows = []
for optimizer_key in ["sgd", "muon"]:
    check = results["training_checks"][optimizer_key]
    training_rows.append([
        check["optimizer_name"],
        check["learning_rate"],
        check["initial_loss"],
        check["final_loss"],
        check["loss_ratio"],
        check["diverged"],
    ])

display(Markdown(markdown_table(
    ["Optimizer", "LR", "Initial loss", "Final loss", "Final/initial", "Diverged?"],
    training_rows,
)))

primary_rows = []
for test_name, test in results["verdict"]["primary_tests"].items():
    primary_rows.append([
        test_name,
        "PASS" if test["pass"] else "FAIL",
        test["description"],
        test["lhs"],
        test["rhs"],
    ])

display(Markdown("### Primary gauge-preserving verdict tests"))
display(Markdown(markdown_table(
    ["Primary test", "Outcome", "Meaning", "lhs", "rhs / threshold"],
    primary_rows,
)))

legacy_rows = []
for test_name, test in results["verdict"]["legacy_tests"].items():
    legacy_rows.append([
        test_name,
        "PASS" if test["pass"] else "FAIL",
        test["description"],
        test["lhs"],
        test["rhs"],
    ])

display(Markdown("### Retained legacy symmetric verdict tests"))
display(Markdown(markdown_table(
    ["Legacy test", "Outcome", "Meaning", "lhs", "rhs / threshold"],
    legacy_rows,
)))

stats = results["statistical_comparison"]
legacy_stats = results["legacy_statistical_comparison"]
display(Markdown(f"""
**Welch-style comparison on primary gauge $\lambda_W$ samples (SGD minus Muon):**
- $t = {stats['t_stat']:.4f}$
- degrees of freedom ≈ {stats['degrees_freedom']:.2f}
- heuristic one-tailed significance flag (`t > 2`): **{stats['heuristic_one_tailed_significant']}**

**Welch-style comparison on retained legacy symmetric $\lambda_W$ samples (SGD minus Muon):**
- $t = {legacy_stats['t_stat']:.4f}$
- degrees of freedom ≈ {legacy_stats['degrees_freedom']:.2f}
- heuristic one-tailed significance flag (`t > 2`): **{legacy_stats['heuristic_one_tailed_significant']}**
"""))

## Representative gauge-preserving trial diagnostics

The representative-trial payload returned by the script is useful for showing that the primary gauge condition really begins with essentially zero effective-map mismatch, even though the raw parameter-space distance $d_W(0)$ is nonzero.

In [ ]:
representative_rows = []
for optimizer_name, condition_key in [("SGD", "SGD_gauge"), ("Muon", "Muon_gauge")]:
    rep = results["condition_results"][condition_key]["representative_trial"]
    representative_rows.append([
        optimizer_name,
        rep["trial_index"],
        rep["seed"],
        rep["initial_effective_mismatch_abs"],
        rep["initial_effective_mismatch_rel"],
        rep["initial_output_mismatch_rel"],
        rep["initial_loss_gap_abs"],
        rep["distance_W"][0],
        rep["distance_F"][0],
    ])

display(Markdown(markdown_table(
    [
        "Optimizer",
        "Trial",
        "Seed",
        "init eff mismatch abs",
        "init eff mismatch rel",
        "init output mismatch rel",
        "init loss gap abs",
        "d_W(0)",
        "d_F(0)",
    ],
    representative_rows,
)))

per_layer_rows = []
for optimizer_name, condition_key in [("SGD", "SGD_gauge"), ("Muon", "Muon_gauge")]:
    rep = results["condition_results"][condition_key]["representative_trial"]
    for layer_idx, layer_lambda in enumerate(rep["lambda_L_per_layer"]):
        per_layer_rows.append([
            optimizer_name,
            rep["trial_index"],
            layer_idx,
            layer_lambda,
            rep["per_layer_distances"][layer_idx, 0],
            rep["per_layer_distances"][layer_idx, -1],
        ])

display(Markdown("### Per-layer finite-time growth for the representative gauge trial"))
display(Markdown(markdown_table(
    ["Optimizer", "Trial", "Layer", "lambda_layer", "d_layer(0)", "d_layer(N)"],
    per_layer_rows,
)))

## Figures produced by the shared script core

The next cells display the PNG files generated by `run_experiment.py`. This keeps the notebook and script aligned numerically and at the level of saved experiment artifacts.

The summary panel now puts the **true gauge-preserving condition first**, followed by the skew control and the retained legacy symmetric comparison.

In [ ]:
figure1 = Path(results["plot_paths"].get("summary_panel", ""))
if figure1 and figure1.exists():
    print(f"Displaying: {figure1}")
    display(Image(filename=str(figure1)))
else:
    display(Markdown("No summary-panel figure was generated."))

### Figure interpretation

- The **gauge** column is the key scientific strengthening: it should show substantial parameter-space displacement while still beginning with effectively zero end-to-end map mismatch.
- The **skew** column is a non-gauge control direction.
- The **legacy symmetric** column makes it visually clear how different the old independent symmetric perturbation is from the new coupled gauge-preserving construction.

For the true gauge condition, any later nonzero $d_F(t)$ is generated by the optimizer dynamics after the common initial function, not by a pre-existing $t=0$ functional mismatch.

In [ ]:
figure2 = Path(results["plot_paths"].get("log_ratio_panel", ""))
if figure2 and figure2.exists():
    print(f"Displaying: {figure2}")
    display(Image(filename=str(figure2)))
else:
    display(Markdown("No log-ratio figure was generated."))

## Explicit caveat: what this pair still does and does not show

This second pass fixes the biggest conceptual flaw of the earlier notebook/script pair, because the primary `gauge` condition is now genuinely function-preserving at initialization.

However, the pair still does **not** establish:
- a quotient-space derivation of a physical tangent direction,
- an asymptotic maximal Lyapunov exponent,
- or robustness across many seeds, widths, depths, and datasets.

So the strongest honest claim is now narrower but more serious:

> the pair can meaningfully test how nearby **gauge-equivalent parameterizations** separate over finite training time under SGD vs. Muon in this toy deep-linear setting.

In [ ]:
verdict = results["verdict"]
primary = verdict["primary_shortform"]
legacy = verdict["legacy_shortform"]
gauge_check = verdict["gauge_construction_check"]
conclusion_md = f"""
## Calibrated conclusion

**Gauge-preserving construction check:** **{gauge_check['pass']}**
- max observed absolute mismatch at $t=0$: **{gauge_check['max_abs_observed']:.3e}**
- max observed relative mismatch at $t=0$: **{gauge_check['max_rel_observed']:.3e}**

**Primary measured outcome:** **{verdict['overall']}** on the true gauge-preserving $\lambda_W$ story ({verdict['primary_tests_passed']}/{verdict['primary_tests_total']} primary tests passed).

On this run:
- primary gauge $\lambda_W$
  - SGD = **{primary['lambda_gauge_SGD']:+.6f}**
  - Muon = **{primary['lambda_gauge_Muon']:+.6f}**
- skew control $\lambda_W$
  - SGD = **{primary['lambda_phys_SGD']:+.6f}**
  - Muon = **{primary['lambda_phys_Muon']:+.6f}**
- retained legacy symmetric $\lambda_W$
  - SGD = **{legacy['lambda_legacy_symmetric_SGD']:+.6f}**
  - Muon = **{legacy['lambda_legacy_symmetric_Muon']:+.6f}**

### Bottom line
This pair now **better supports the intended gauge-direction sensitivity story at the level of experimental construction**, because the primary `gauge` perturbation really is function-preserving at initialization.

But whether it **supports the stability hypothesis itself** still depends on the measured result above. In this run, the correct conclusion is:

> **{verdict['interpretation']}**

So the second pass improves the scientific honesty and relevance of the pair substantially, even if the observed numerical result may still be mixed or negative for the original Muon-stability claim.
"""

display(Markdown(conclusion_md))